In [3]:
import UNet.functions.SinkTools2 as st
import time
from osgeo import gdal, gdalconst
start_time = time.time()

# Insert Raster Path & Output Directory, Define Tile Size & Overlap

In [1]:
# input_DSM_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/DSM_crop_-32767.tif'
input_DSM_path = '/Users/nirgut/Documents/Master_Files/UNet_files/Base_Rasters/DSM_new_boundaries.tif'
output_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/all/'
output_notempty_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/not_empty'
output_empty_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/empty'
# input_mask_path = '/Users/nirgut/Documents/Master_Files/pipeline_testing/Mask_crop_-32767.tif'
input_mask_path = '/Users/nirgut/Documents/Master_Files/UNet_files/Base_Rasters/Mask_new_boundaries.tif'
output_mask_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/all/'
output_notempty_mask_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/not_empty'

DSM_norm_empty = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlapn/DSM_tiles/normalized/empty'
DSM_norm_notempty = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/not_empty'

# Define the size of the tiles
tile_size_x = 256  # width of each tile in pixels
tile_size_y = 256  # height of each tile in pixels
overlap_x = 200  # Example overlap in x direction
overlap_y = 200  # Example overlap in y direction

In [4]:
# creating tiles
st.divide_raster_to_tiles(input_DSM_path, output_DSM_tiles_dir, tile_size_x, tile_size_y, overlap_x, overlap_y)

st.divide_raster_to_tiles(input_mask_path, output_mask_tiles_dir, tile_size_x, tile_size_y, overlap_x, overlap_y)

100%|██████████| 618/618 [1:48:51<00:00, 10.57s/it]


# Filtering tiles by mask

In [5]:
st.check_empty_rasters(output_mask_tiles_dir, 10, output_DSM_tiles_dir)

The size of the directory is: 1321945


100%|██████████| 1321945/1321945 [2:09:54<00:00, 169.61it/s] 


In [19]:
import numpy as np
import os

def normalize_raster(input_raster_path, output_raster_path):
    """
    Normalize the values of a raster to the range 0-1 and save it as a new raster.

    :param input_raster_path: Path to the input raster file.
    :param output_raster_path: Path to the output raster file.
    """
    # Open the input raster
    dataset = gdal.Open(input_raster_path, gdalconst.GA_ReadOnly)
    if not dataset:
        print(f"Failed to open raster file: {input_raster_path}")
        return

    # Read the raster data
    band = dataset.GetRasterBand(1)
    array = band.ReadAsArray()

    # Normalize the array to the range 0-1
    min_val = np.min(array)
    max_val = np.max(array)
    norm_array = (array - min_val) / (max_val - min_val)

    # Create the output raster
    driver = gdal.GetDriverByName('GTiff')
    out_dataset = driver.Create(output_raster_path, dataset.RasterXSize, dataset.RasterYSize, 1, gdal.GDT_Float32)
    if not out_dataset:
        print(f"Failed to create output raster file: {output_raster_path}")
        return
    
    out_dataset.SetGeoTransform(dataset.GetGeoTransform())
    out_dataset.SetProjection(dataset.GetProjection())

    # Write the normalized data to the output raster
    out_band = out_dataset.GetRasterBand(1)
    out_band.WriteArray(norm_array)
    out_band.SetNoDataValue(-32767)

    # Clean up
    band = None
    dataset = None
    out_band = None
    out_dataset = None

def normalize_rasters_in_folder(input_folder, output_folder):
    """
    Normalize the values of all rasters in a folder to the range 0-1 and save them as new rasters.

    :param input_folder: Path to the folder containing input rasters.
    :param output_folder: Path to the folder where output rasters will be saved.
    """
    # Check if the output folder exists, create it if not
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Process each raster file in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith('.tif'):
            input_raster_path = os.path.join(input_folder, filename)
            output_raster_path = os.path.join(output_folder, filename)
            print(f"Normalizing {input_raster_path} and saving to {output_raster_path}")
            normalize_raster(input_raster_path, output_raster_path)





In [20]:
output_notempty_DSM_tiles_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train'
DSM_norm_notempty = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train_normalized'
normalize_rasters_in_folder(output_notempty_DSM_tiles_dir, DSM_norm_notempty)
# normalize_rasters_in_folder(output_empty_DSM_tiles_dir, DSM_norm_empty)

Normalizing /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train/tile_117_1802.tif and saving to /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train_normalized/tile_117_1802.tif
Normalizing /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train/tile_140_2024.tif and saving to /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train_normalized/tile_140_2024.tif
Normalizing /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train/tile_116_1816.tif and saving to /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train_normalized/tile_116_1816.tif
Normalizing /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overla

In [ ]:
st.split_data_and_masks(DSM_norm_notempty, output_notempty_mask_tiles_dir, 0.8)

In [ ]:
end_time = time.time()
print("Total time: ", end_time - start_time)


## Creating a shapefile from the tiles

In [16]:
import os
import geopandas as gpd
from shapely.geometry import box
import rasterio
import tqdm

# Specify the directory containing the raster tiles
raster_dir = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/train_random3500'

# Create a list to store the geometries and filenames
data = []

# Iterate through each raster file in the directory
for filename in tqdm.tqdm(os.listdir(raster_dir)):
    if filename.endswith('.tif'):
        raster_path = os.path.join(raster_dir, filename)

        # Open the raster file to get its bounds
        with rasterio.open(raster_path) as src:
            # Get the bounds of the raster
            bounds = src.bounds
            # Create a bounding box geometry
            geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            # Append the geometry and filename to the data list
            data.append({'geometry': geom, 'filename': filename})
            


# Create a GeoDataFrame from the data
gdf = gpd.GeoDataFrame(data, crs='EPSG:2039')  # Use appropriate CRS for your data

# Specify the output shapefile path
output_shapefile = '/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/train_random3000.shp'

# Save the GeoDataFrame to a shapefile
gdf.to_file(output_shapefile, driver='ESRI Shapefile')

print(f'Shapefile saved at: {output_shapefile}')


100%|██████████| 3500/3500 [00:10<00:00, 333.79it/s]


Shapefile saved at: /Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/train_random3000.shp


## Copying the raster files to a new directory based on a shapefile

In [18]:
import geopandas as gpd
import os
import shutil

# Input shapefile and directories
shapefile_path = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/train_random3000.shp"
raster_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/all_train"
output_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/random3000_train"

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# Load the shapefile
gdf = gpd.read_file(shapefile_path)

# Extract the filenames from the "filename" field
shape_filenames = set(gdf["filename"])

# Iterate through files in the raster directory
for raster_file in os.listdir(raster_directory):
    # Check if the file is in the shapefile list
    if raster_file in shape_filenames:
        # Copy the file to the output directory
        shutil.copy(os.path.join(raster_directory, raster_file), output_directory)

print("Files copied successfully!")


Files copied successfully!


## Randomly selecting 3000 files from a shapefile and copying them to a new directory

In [15]:
import geopandas as gpd
import os
import shutil
import random

# Input shapefile and directories
shapefile_path = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/all_train_tiles.shp"
mask_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/all_train"
dsm_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/all_train"
output_mask_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/Mask_tiles/train_random3500"
output_dsm_directory = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/extra_tiles_200overlap/DSM_tiles/train_random3500"

# Create the output directories if they don't exist
os.makedirs(output_mask_directory, exist_ok=True)
os.makedirs(output_dsm_directory, exist_ok=True)

# Load the shapefile
gdf = gpd.read_file(shapefile_path)

# Check if there are at least 3000 features
if len(gdf) < 3500:
    raise ValueError("The shapefile contains fewer than 3000 features.")

# Randomly select 3000 features
random_features = gdf.sample(n=3500, random_state=42)  # Set random_state for reproducibility

# Extract the filenames from the selected features
selected_filenames = set(random_features["filename"])

# Copy files from the mask directory
for mask_file in os.listdir(mask_directory):
    if mask_file in selected_filenames:
        shutil.copy(os.path.join(mask_directory, mask_file), output_mask_directory)

# Copy files from the DSM directory
for dsm_file in os.listdir(dsm_directory):
    if dsm_file in selected_filenames:
        shutil.copy(os.path.join(dsm_directory, dsm_file), output_dsm_directory)

print("3000 random files copied successfully for both mask and DSM directories!")



3000 random files copied successfully for both mask and DSM directories!


In [21]:
import os
import shutil

# Define the directories
source_folder = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/128_overlap/Mask_tiles/not_empty"
reference_folder = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/128_overlap/DSM_tiles/train_not_empty"
destination_folder = "/Users/nirgut/Documents/Master_Files/UNet_files/Train_Test_Data/128_overlap/Mask_tiles/train_not_empty"

# Create the destination folder if it does not exist
os.makedirs(destination_folder, exist_ok=True)

# Get the filenames from the reference folder
reference_filenames = set(os.listdir(reference_folder))

# Iterate through the files in the source folder
for file in os.listdir(source_folder):
    if file in reference_filenames:  # Check if the file exists in the reference folder
        # Copy the file to the destination folder
        shutil.copy(os.path.join(source_folder, file), destination_folder)

print("Files successfully copied based on reference folder!")


Files successfully copied based on reference folder!
